In [13]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import tqdm
import pandas as pd
import matplotlib.pyplot as plt

from torch_geometric.data import Data, Dataset, Batch
from torch_geometric.nn import NNConv, global_mean_pool

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [12]:
import os 
dir = os.getcwd()
leakdb = os.path.join(dir, "..")
text = os.path.join(leakdb, "txt")


In [14]:
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [15]:
def load_scenario_data(scenario_id, base_path=r"D:\LeakDB_full_data\Hanoi"):
    try:
        scenario_path = os.path.join(base_path, f"Scenario-{scenario_id}")
        if not os.path.exists(scenario_path):
            print(f"Scenario {scenario_id} not found.")
            return None
        
        leaks, timestamps = None, None
        demand_path = flow_path = pressure_path = None
        for sub in os.listdir(scenario_path):
            if sub in [f"Scenario-{scenario_id}", f"Scenario-{scenario_id}_info.csv", f"Hanoi_CMH_Scenario-{scenario_id}.inp"]:
                continue
            sub_path = os.path.join(scenario_path, sub)
            if sub == "Demands":
                demand_path = sub_path
            elif sub == "Flows":
                flow_path = sub_path
            elif sub == "Pressures":
                pressure_path = sub_path
            elif sub == "Labels.csv":
                leaks = pd.read_csv(sub_path).drop(columns=["Index"], errors="ignore")
            elif sub == "Timestamps.csv":
                timestamps = pd.read_csv(sub_path).drop(columns=["Index"], errors="ignore")

        if not all([demand_path, flow_path, pressure_path, leaks is not None, timestamps is not None]):
            print(f"Scenario {scenario_id} is incomplete.")
            return None

        df = pd.concat([leaks, timestamps], axis=1, ignore_index=True)
        df.columns = ["Leaks", "Timestamps"]

        def combined_feature_df(path, feature):
            dfs = []
            for file in sorted(os.listdir(path)):
                file_path = os.path.join(path, file)
                if not file.endswith(".csv"):
                    continue
                sub_df = pd.read_csv(file_path).drop(columns="Index", errors="ignore")
                sub_df.columns = [f"{feature}_{file.split('.')[0]}"]
                dfs.append(sub_df)
            return pd.concat(dfs, axis=1, ignore_index=True)

        demand_df = combined_feature_df(demand_path, "demand")
        pressure_df = combined_feature_df(pressure_path, "pressure")
        flow_df = combined_feature_df(flow_path, "flow")

        demand_df.columns = [f"demand_node_{i}" for i in range(1, demand_df.shape[1] + 1)]
        pressure_df.columns = [f"pressure_node_{i}" for i in range(1, pressure_df.shape[1] + 1)]
        flow_df.columns = [f"flow_link_{i}" for i in range(1, flow_df.shape[1] + 1)]

        final_df = pd.concat([demand_df, pressure_df, flow_df, df], axis=1)
        final_df["Leaks"] = final_df["Leaks"].astype(int)
        return final_df

    except Exception as e:
        print(f"Error loading scenario {scenario_id}: {e}")
        return None

In [16]:
def aggregate_nonleak_rows(df, leak_col="Leaks"):
    data_cols = [c for c in df.columns if c != "Timestamps" and c != leak_col]
    out_rows = []
    i = 0
    n = len(df)
    while i < n:
        if df.iloc[i][leak_col] == 1:
            out_rows.append(df.iloc[i][data_cols + [leak_col]].to_dict())
            i += 1
            continue
        j = i
        while j < n and df.iloc[j][leak_col] == 0:
            j += 1
        run_len = j - i
        k = i
        while k + 3 <= j:
            window = df.iloc[k:k+3]
            avg_vals = window[data_cols].mean(axis=0).to_dict()
            avg_vals[leak_col] = 0
            out_rows.append(avg_vals)
            k += 3
        while k < j:
            out_rows.append(df.iloc[k][data_cols + [leak_col]].to_dict())
            k += 1
        i = j
    out_df = pd.DataFrame(out_rows)
    return out_df

In [17]:
def compute_global_mean_std(df_list):
    sum_ = None
    sumsq_ = None
    n_total = 0
    for df in df_list:
        arr = df.drop(columns=["Leaks"]).values.astype(np.float64)
        if sum_ is None:
            sum_ = arr.sum(axis=0)
            sumsq_ = (arr**2).sum(axis=0)
        else:
            sum_ += arr.sum(axis=0)
            sumsq_ += (arr**2).sum(axis=0)
        n_total += arr.shape[0]
    mean = sum_ / n_total
    var = (sumsq_ / n_total) - (mean**2)
    std = np.sqrt(np.maximum(var, 1e-6))
    return mean.astype(np.float32), std.astype(np.float32)

def normalize_df(df, mean, std):
    cols = [c for c in df.columns if c != "Leaks"]
    df2 = df.copy()
    df2[cols] = (df2[cols] - mean) / std
    return df2

In [18]:
def load_edge_index(pipes_path):
    
    edges = []
    
    with open(pipes_path, "r") as file:
        for line in file:
            if line.strip().startswith(";") or not line.strip():
                continue
            
            parts = line.split()
            if len(parts) < 3:
                continue
            
            try:
                n1, n2 = int(parts[1]), int(parts[2])
                edges.append([n1 - 1, n2 - 1])
            except:
                continue
            
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index

In [19]:
def build_graph_from_row(row, edge_index, num_nodes=32):
    
    node_feats = []
    for i in range(num_nodes):
        node_feats.append([
            float(row.get(f"demand_node_{i}", 0.0)),
            float(row.get(f"pressure_node_{i}", 0.0))
        ])
        
    x = torch.tensor(node_feats, dtype=torch.float)
    
    num_edges = edge_index.shape[1]
    edge_feats = []
    for i in range(num_edges):
        col_name = f"flow_link_{i+1}"
        edge_feats.append([float(row.get(col_name, 0.0))])
    edge_attr = torch.tensor(edge_feats, dtype=torch.float)

    y = torch.tensor([int(row.get("Leaks", 0))], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

In [20]:
class ScenarioGraphSequenceDataset(Dataset):
    def __init__(self, df_list, edge_index, num_nodes=32, seq_len=5):
        super().__init__()
        self.samples = []
        self.edge_index = edge_index
        self.num_nodes = num_nodes
        self.seq_len = seq_len

        for df in tqdm.tqdm(df_list, desc="Building ST sequences"):
            for i in range(len(df) - seq_len + 1):
                seq_df = df.iloc[i:i + seq_len]
                label = float(seq_df.iloc[-1]["Leaks"])
                seq_graphs = []
                for _, row in seq_df.iterrows():
                    g = build_graph_from_row(row, edge_index, num_nodes)
                    seq_graphs.append(g)
                self.samples.append((seq_graphs, label))
    
    def len(self):
        return len(self.samples)

    def get(self, idx):
        seq_graphs, label = self.samples[idx]
        return seq_graphs, torch.tensor(label, dtype=torch.float32)

In [21]:
def stgnn_collate_fn(batch):
    seqs, labels = zip(*batch) 
    seq_len = len(seqs[0])
    batch_by_timestep = []
    for t in range(seq_len):
        batch_by_timestep.append([seq[t] for seq in seqs])
    labels = torch.tensor([lbl.item() if isinstance(lbl, torch.Tensor) else float(lbl) for lbl in labels], dtype=torch.float32)
    return batch_by_timestep, labels

In [22]:
class LeakSTGNN(nn.Module):
    def __init__(self, node_in=2, edge_in=1, hidden_gnn=64, hidden_gru=32, dropout=0.3):
        super().__init__()
        nn_edge = nn.Sequential(nn.Linear(edge_in, 16), nn.ReLU(), nn.Linear(16, node_in * hidden_gnn))
        self.conv = NNConv(node_in, hidden_gnn, nn_edge, aggr='mean')
        self.gru = nn.GRU(hidden_gnn, hidden_gru, batch_first=True)
        self.fc = nn.Linear(hidden_gru, 1)
        self.dropout = dropout

    def forward(self, x_list, edge_index_list, edge_attr_list, batch_list):
        gnn_outputs = []
        for x, edge_index, edge_attr, batch in zip(x_list, edge_index_list, edge_attr_list, batch_list):

            if edge_attr.shape[0] != edge_index.shape[1]:
                raise RuntimeError(f"edge_attr rows {edge_attr.shape[0]} != edge_index columns {edge_index.shape[1]}")

            xh = F.relu(self.conv(x, edge_index, edge_attr))
            pooled = global_mean_pool(xh, batch)
            gnn_outputs.append(pooled.unsqueeze(1))
        gnn_seq = torch.cat(gnn_outputs, dim=1) 
        _, h = self.gru(gnn_seq)                
        out = torch.sigmoid(self.fc(h[-1])).squeeze(1)
        return out


In [24]:
ALL_SCENARIOS = [82, 15, 4, 95, 36, 32, 29, 18, 14, 87, 70, 12, 76, 55, 5, 98, 89, 28, 30, 65, 78, 85, 72, 26, 90, 54, 94, 58, 96, 1, 21, 91, 44, 80, 20, 83, 68, 7, 6, 25, 63, 23, 69, 59, 39, 17, 52, 3, 47, 100, 35, 8, 61, 62, 67, 19, 41, 40, 24, 37, 13, 86, 53, 99, 45, 75, 64, 60, 48, 9, 34, 88, 27, 84, 50, 81, 97, 33, 22, 31, 38, 77, 93, 49, 46, 73, 57, 56, 11, 74, 79, 16, 71, 2, 51, 66, 10, 92, 42, 43]

ALL_SCENARIOS = random.sample(ALL_SCENARIOS, 50)
ALL_SCENARIOS
len(ALL_SCENARIOS)
# TRAIN_SCENARIOS = ALL_SCENARIOS[:70]
# VAL_SCENARIOS = ALL_SCENARIOS[70:85]
# TEST_SCENARIOS = ALL_SCENARIOS[85:100]

50

In [25]:
TRAIN_SCENARIOS = ALL_SCENARIOS[:35]
VAL_SCENARIOS = ALL_SCENARIOS[35:42]
TEST_SCENARIOS = ALL_SCENARIOS[42:50]

In [26]:
def train_stgnn_pipeline(
    pipes_path,
    base_path,
    seq_len=5,
    epochs=15,
    batch_size=8,
    lr=1e-3,
    dropout=0.3,
    device=None,
    num_nodes=32
):
    
    mean_path = os.path.join(text, "mean_stgnn.txt")
    std_path = os.path.join(text, "std_stgnn.txt")
    model_path = os.path.join(leakdb, "best_leak_stgnn.pth")

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dfs, val_dfs, test_dfs = [], [], []

    for sid in tqdm.tqdm(TRAIN_SCENARIOS, desc="Loading scenarios [Train]"):
        df = load_scenario_data(sid, base_path)
        if df is None or len(df) == 0:
            continue
        if "Timestamps" in df.columns:
            df = df.drop(columns=["Timestamps"])
        #df = aggregate_nonleak_rows(df)
        df.reset_index(drop=True, inplace=True)
        train_dfs.append(df)
        
    for sid in tqdm.tqdm(VAL_SCENARIOS, desc="Loading scenarios [Val]"):
        df = load_scenario_data(sid, base_path)
        if df is None or len(df) == 0:
            continue
        if "Timestamps" in df.columns:
            df = df.drop(columns=["Timestamps"])
        #df = aggregate_nonleak_rows(df)
        df.reset_index(drop=True, inplace=True)
        val_dfs.append(df)
        
    for sid in tqdm.tqdm(TEST_SCENARIOS, desc="Loading scenarios [Test]"):
        df = load_scenario_data(sid, base_path)
        if df is None or len(df) == 0:
            continue
        if "Timestamps" in df.columns:
            df = df.drop(columns=["Timestamps"])
        #df = aggregate_nonleak_rows(df)
        df.reset_index(drop=True, inplace=True)
        test_dfs.append(df)

    mean, std = compute_global_mean_std(train_dfs)
    np.savetxt(mean_path, mean)
    np.savetxt(std_path, std)

    train_dfs = [normalize_df(df, mean, std) for df in train_dfs]
    val_dfs   = [normalize_df(df, mean, std) for df in val_dfs]
    test_dfs  = [normalize_df(df, mean, std) for df in test_dfs]

    edge_index_global = load_edge_index(pipes_path)
    train_set = ScenarioGraphSequenceDataset(train_dfs, edge_index_global, num_nodes=num_nodes, seq_len=seq_len)
    val_set   = ScenarioGraphSequenceDataset(val_dfs, edge_index_global, num_nodes=num_nodes, seq_len=seq_len)
    test_set  = ScenarioGraphSequenceDataset(test_dfs, edge_index_global, num_nodes=num_nodes, seq_len=seq_len)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=stgnn_collate_fn)
    val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=batch_size, shuffle=False, collate_fn=stgnn_collate_fn)
    test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=batch_size, shuffle=False, collate_fn=stgnn_collate_fn)

    model = LeakSTGNN(node_in=2, edge_in=1, hidden_gnn=64, hidden_gru=32, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCELoss()

    best_val_f1 = 0.0
    history = {"train_loss": [], "val_loss": [], "val_f1": []}
    model_path = model_path

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for batch_by_timestep, labels in train_loader:
          
            x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
            for seq_t in batch_by_timestep:
                batch = Batch.from_data_list(seq_t).to(device)
          
                if batch.edge_attr.shape[0] != batch.edge_index.shape[1]:
                    raise RuntimeError(f"Edge mismatch in batched timestep: edge_attr {batch.edge_attr.shape} vs edge_index {batch.edge_index.shape}")
                x_list.append(batch.x)
                edge_index_list.append(batch.edge_index)
                edge_attr_list.append(batch.edge_attr)
                batch_list.append(batch.batch)

            labels = labels.to(device)
            preds = model(x_list, edge_index_list, edge_attr_list, batch_list)
            loss = criterion(preds, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        history["train_loss"].append(train_loss)

        model.eval()
        val_loss = 0.0
        preds_all, labs_all = [], []
        with torch.no_grad():
            for batch_by_timestep, labels in val_loader:
                x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
                for seq_t in batch_by_timestep:
                    batch = Batch.from_data_list(seq_t).to(device)
                    x_list.append(batch.x)
                    edge_index_list.append(batch.edge_index)
                    edge_attr_list.append(batch.edge_attr)
                    batch_list.append(batch.batch)
                labels = labels.to(device)
                preds = model(x_list, edge_index_list, edge_attr_list, batch_list)
                loss = criterion(preds, labels)
                val_loss += loss.item()
                preds_all.append(preds.cpu().numpy())
                labs_all.append(labels.cpu().numpy())

        val_loss /= len(val_loader)
        preds_all = np.concatenate(preds_all) if len(preds_all) > 0 else np.array([])
        labs_all  = np.concatenate(labs_all) if len(labs_all) > 0 else np.array([])
        if preds_all.size and labs_all.size:
            pred_labels = (preds_all >= 0.5).astype(int)
            p, r, f1, _ = precision_recall_fscore_support(labs_all, pred_labels, average="binary", zero_division=0)
        else:
            p = r = f1 = 0.0

        history["val_loss"].append(val_loss)
        history["val_f1"].append(f1)
        print(f"Epoch {epoch:02d}/{epochs} TrainLoss={train_loss:.4f} ValLoss={val_loss:.4f} ValF1={f1:.4f}")

        if f1 > best_val_f1:
            best_val_f1 = f1
            torch.save(model.state_dict(), model_path)

    print(f"\nTraining complete. Best Val F1: {best_val_f1:.4f}")

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    preds_all, labs_all = [], []
    with torch.no_grad():
        for batch_by_timestep, labels in test_loader:
            x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
            for seq_t in batch_by_timestep:
                batch = Batch.from_data_list(seq_t).to(device)
                x_list.append(batch.x)
                edge_index_list.append(batch.edge_index)
                edge_attr_list.append(batch.edge_attr)
                batch_list.append(batch.batch)
            labels = labels.to(device)
            preds = model(x_list, edge_index_list, edge_attr_list, batch_list)
            preds_all.append(preds.cpu().numpy())
            labs_all.append(labels.cpu().numpy())

    preds_all = np.concatenate(preds_all)
    labs_all  = np.concatenate(labs_all)
    pred_labels = (preds_all >= 0.5).astype(int)
    acc = accuracy_score(labs_all, pred_labels)
    p, r, f1, _ = precision_recall_fscore_support(labs_all, pred_labels, average="binary", zero_division=0)
    cm = confusion_matrix(labs_all, pred_labels)

    print(f"\nTest Metrics:\nAcc={acc:.4f}  Precision={p:.4f}  Recall={r:.4f}  F1={f1:.4f}")
    print("Confusion Matrix:\n", cm)

    plt.figure(figsize=(5,4))
    plt.imshow(cm, cmap='Blues', interpolation='nearest')
    plt.title("Test Confusion Matrix (STGNN)")
    plt.xlabel("Predicted"); plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i,j], ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
    plt.tight_layout()
    plt.show()

    return model, history, (acc, p, r, f1)

In [18]:
mean_path = os.path.join(text, "mean_stgnn.txt")
std_path = os.path.join(text, "std_stgnn.txt")
model_path = os.path.join(leakdb, "best_leak_stgnn.pth")
pipes_path = os.path.join(text, "inp_1_text.txt")

In [19]:
def test_unseen_scenario_stgnn(scenario_id, pipes_path, base_path, model_path=model_path, mean_path=mean_path, std_path=std_path, seq_len=5, device=None, num_nodes=32):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    mean = np.loadtxt(mean_path)
    std = np.loadtxt(std_path)

    df = load_scenario_data(scenario_id, base_path)
    if df is None or len(df) == 0:
        print(f"Scenario {scenario_id} could not be loaded.")
        return None
    if "Timestamps" in df.columns:
        df = df.drop(columns=["Timestamps"])
    df = df.reset_index(drop=True)
    df = normalize_df(df, mean, std)

    edge_index = load_edge_index(pipes_path)
    dataset = ScenarioGraphSequenceDataset([df], edge_index, num_nodes=num_nodes, seq_len=seq_len)
    loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False, collate_fn=stgnn_collate_fn)

    model = LeakSTGNN(node_in=2, edge_in=1, hidden_gnn=64, hidden_gru=32).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    preds_all, labs_all = [], []
    with torch.no_grad():
        for batch_by_timestep, labels in loader:
            x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
            for seq_t in batch_by_timestep:
                batch = Batch.from_data_list(seq_t).to(device)
                x_list.append(batch.x)
                edge_index_list.append(batch.edge_index)
                edge_attr_list.append(batch.edge_attr)
                batch_list.append(batch.batch)
            labels = labels.to(device)
            preds = model(x_list, edge_index_list, edge_attr_list, batch_list)
            preds_all.append(preds.cpu().numpy())
            labs_all.append(labels.cpu().numpy())

    preds_all = np.concatenate(preds_all)
    labs_all  = np.concatenate(labs_all)
    pred_labels = (preds_all >= 0.5).astype(int)
    acc = accuracy_score(labs_all, pred_labels)
    p, r, f1, _ = precision_recall_fscore_support(labs_all, pred_labels, average="binary", zero_division=0)
    cm = confusion_matrix(labs_all, pred_labels)

    print(f"\nScenario {scenario_id} Results:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {p:.4f}")
    print(f"Recall:    {r:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)

    plt.figure(figsize=(4,4))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(f"Confusion Matrix - Scenario {scenario_id}")
    plt.colorbar()
    plt.xlabel("Predicted"); plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
    plt.show()

    return {"scenario_id": scenario_id, "accuracy": acc, "precision": p, "recall": r, "f1": f1, "confusion_matrix": cm}

In [16]:
def test_single_sequence(
    model_path,
    pipes_path,
    scenario_df,
    mean_path,
    std_path,
    seq_len=5,
    idx=None,
    num_nodes=32,
    device=None
):
    
    import torch
    import numpy as np
    from torch_geometric.data import Batch
    
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    mean = np.loadtxt(mean_path)
    std = np.loadtxt(std_path)
    
    df = scenario_df.copy()
    
    if "Timestamps" in df.columns:
        df = df.drop(columns=["Timestamps"])
        
    df = aggregate_nonleak_rows(df)
    df = normalize_df(df, mean, std)
    df.reset_index(drop=True, inplace=True)
    
    if idx is None:
        idx = len(df) - 1
    
    if idx - seq_len + 1 < 0:
        raise ValueError(f"Not enough history before index {idx} for seq_len={seq_len}")
    
    seq_df = df.iloc[idx - seq_len + 1 : idx + 1]
    print(f"\n Using rows {idx} = {seq_len + 1} -> {idx} for prediction (window length {seq_len})")
    
    edge_index = load_edge_index(pipes_path)
    seq_graphs = [build_graph_from_row(row, edge_index, num_nodes) for _, row in seq_df.iterrows()]
    
    x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
    for g in seq_graphs:
        batch = Batch.from_data_list([g]).to(device)
        x_list.append(batch.x)
        edge_index_list.append(batch.edge_index)
        edge_attr_list.append(batch.edge_attr)
        batch_list.append(batch.batch)
        
    model = LeakSTGNN().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    with torch.no_grad():
        pred = model(x_list, edge_index_list, edge_attr_list, batch_list)
        pred_val = pred.item()
        
    true_val = seq_df.iloc[-1]["Leaks"]
    
    print(f"Prediction (Leak Probability): {pred_val:.4f}")
    print(f"True Label:{true_val}")
    print("Prediction:", "LEAK " if pred_val >= 0.5 else "NO LEAK ")
    
    return float(pred_val), int(true_val)

In [23]:
pipes_path = os.path.join(text, "inp_1_text.txt")
base_path = r"D:\LeakDB_full_data\Hanoi"

df = load_scenario_data(54, base_path)
pred, truth = test_single_sequence(
    model_path=os.path.join(leakdb, "best_leak_stgnn.pth"),
    pipes_path=pipes_path,
    scenario_df=df,
    mean_path=mean_path,
    std_path=std_path,
    idx=12004  # will use [idx - 4 .... idx]
)

print(pred, truth)



 Using rows 12004 = 6 -> 12004 for prediction (window length 5)
Prediction (Leak Probability): 0.9859
True Label:1.0
Prediction: LEAK 
0.9858786463737488 1


In [21]:
def detect_and_localize(
    model_path, pipes_path, scenario_df,
    mean_path, std_path,
    seq_len=5, start_idx=4, threshold=0.5, num_nodes=32,
    max_no_leaks=10, device=None
):
    
    import torch, numpy as np
    from torch_geometric.data import Batch
    from scipy.stats import zscore
    
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    mean, std = np.loadtxt(mean_path), np.loadtxt(std_path)
    
    df = scenario_df.copy()
    
    if "Timestamps" in df.columns:
        df = df.drop(columns=["Timestamps"])
        
    df = aggregate_nonleak_rows(df) # only for saving time
    scenario_df = aggregate_nonleak_rows(scenario_df) # only to maintain consistency for both
    df = normalize_df(df, mean, std)
    df.reset_index(drop=True, inplace=True)
    
    edge_index = load_edge_index(pipes_path)
    model = LeakSTGNN().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    leak_detected = False
    no_leak_counter = 0
    leak_nodes = []
    probs = []
    
    for idx in range(start_idx, len(df)):
        
        if idx - seq_len + 1 < 0:
            continue
    
        seq_df = df.iloc[idx - seq_len + 1: idx + 1]
        seq_df_original = scenario_df.iloc[idx - seq_len + 1: idx + 1]
        seq_graphs = [build_graph_from_row(row, edge_index, num_nodes) for _, row in seq_df.iterrows()]
        
        x_list, edge_index_list, edge_attr_list, batch_list = [], [], [], []
        
        for g in seq_graphs:
            batch = Batch.from_data_list([g]).to(device)
            x_list.append(batch.x)
            edge_index_list.append(batch.edge_index)
            edge_attr_list.append(batch.edge_attr)
            batch_list.append(batch.batch)

        with torch.no_grad():
            p = model(x_list, edge_index_list, edge_attr_list, batch_list).item()
        probs.append(p)

        if p >= threshold:
            if not leak_detected:
                print(f"Leak detected at index {idx} (prob={p})")
                leak_detected = True
                no_leak_counter = 0
                
            pressure_cols = [f"pressure_node_{i+1}" for i in range(num_nodes)
                             if f"pressure_node_{i+1}" in seq_df.columns]
            if pressure_cols:
                node_std = seq_df_original[pressure_cols].std().values
                leak_node = int(np.argmax(node_std)) + 1
                leak_nodes.append(leak_node)
                print(f"   → Highest deviation at node {leak_node}")
                
        else:
            if leak_detected:
                no_leak_counter += 1
                
                if no_leak_counter >= max_no_leaks:
                    print(f"Leak ended around index {idx}")
                    leak_detected = False
                    no_leak_counter = 0
                    
    if leak_nodes:
        final_node = max(set(leak_nodes), key=leak_nodes.count)
        print(f"Most likely leak node:{final_node}")
    else:
        print("No leaks detected in this scenario.")
        
    return probs, leak_nodes

In [22]:
test_df = load_scenario_data(434)

# Scenario-434: leak nodes: [25]

probs, leak_nodes = detect_and_localize(
    model_path=model_path,
    pipes_path=os.path.join(text, "inp_1_text.txt"),
    scenario_df=test_df,
    mean_path=mean_path,
    std_path=std_path    
)

Leak detected at index 92 (prob=0.5619663000106812)
   → Highest deviation at node 15
Leak ended around index 102
Leak detected at index 155 (prob=0.6460910439491272)
   → Highest deviation at node 15
Leak ended around index 165
Leak detected at index 205 (prob=0.8589604496955872)
   → Highest deviation at node 15
Leak ended around index 215
Leak detected at index 317 (prob=0.8848756551742554)
   → Highest deviation at node 15
Leak ended around index 327
Leak detected at index 381 (prob=0.9370337128639221)
   → Highest deviation at node 24
Leak ended around index 391
Leak detected at index 397 (prob=0.5129509568214417)
   → Highest deviation at node 15
Leak ended around index 407
Leak detected at index 539 (prob=0.7034381031990051)
   → Highest deviation at node 15
   → Highest deviation at node 24
Leak ended around index 550
Leak detected at index 637 (prob=0.5397891402244568)
   → Highest deviation at node 24
Leak ended around index 647
Leak detected at index 651 (prob=0.589960098266

In [18]:
test_df = load_scenario_data(52)

#Scenario-52: leak nodes: [18]

probs, leak_nodes = detect_and_localize(
    model_path=model_path,
    pipes_path=os.path.join(text, "inp_1_text.txt"),
    scenario_df=test_df,
    mean_path=mean_path,
    std_path=std_path    
)

Leak detected at index 61 (prob=0.5840854048728943)
   → Highest deviation at node 16
Leak ended around index 71
Leak detected at index 189 (prob=0.5903642177581787)
   → Highest deviation at node 16
Leak ended around index 199
Leak detected at index 315 (prob=0.8788884282112122)
   → Highest deviation at node 16
   → Highest deviation at node 16
Leak ended around index 326
Leak detected at index 539 (prob=0.6821526288986206)
   → Highest deviation at node 16
   → Highest deviation at node 16
   → Highest deviation at node 16
Leak ended around index 551
Leak detected at index 653 (prob=0.619590163230896)
   → Highest deviation at node 16
Leak ended around index 663
Leak detected at index 714 (prob=0.529861330986023)
   → Highest deviation at node 16
Leak ended around index 724
Leak detected at index 730 (prob=0.581954300403595)
   → Highest deviation at node 16
Leak ended around index 740
Leak detected at index 749 (prob=0.5491935610771179)
   → Highest deviation at node 5
Leak ended a